# Solutions — Diagnostic 4 — Functions & OOP

Exercise notebook: `00_diagnostic/diagnostic_04_functions_oop.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Question 1 — 🟢 Easy · An accumulator closure

**Working solution**

In [ ]:
def make_accumulator(start=0):
    total = start
    def add(x):
        nonlocal total
        total += x
        return total
    return add

**Explanation**

Closure + `nonlocal`.

**Complexity:** O(1).

**Common mistakes**

- Global state.

## Question 2 — 🟡 Medium · Properties with validation

**Working solution**

In [ ]:
from functools import total_ordering

@total_ordering
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

    def __eq__(self, other):
        return isinstance(other, Temperature) and self.celsius == other.celsius

    def __lt__(self, other):
        return self.celsius < other.celsius

    def __repr__(self):
        return f"Temperature({self.celsius}°C)"

**Explanation**

Properties keep attribute syntax while adding validation and derived values; `total_ordering` fills in comparisons.

**Complexity:** O(1).

**Common mistakes**

- `getCelsius()`-style methods.
- Bypassing the setter in `__init__`.

## Question 3 — 🟡 Medium · A call-counting decorator

**Working solution**

In [ ]:
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

**Explanation**

Decorator = function returning a wrapper; `functools.wraps` preserves metadata; function attributes hold state.

**Complexity:** O(1).

**Common mistakes**

- Missing `wraps`.
- Counting only successful calls.

## Question 4 — 🟠 Hard · An inventory with dunders and a transactional context manager

**Working solution**

In [ ]:
from contextlib import contextmanager

class Inventory:
    def __init__(self):
        self._stock = {}

    def add(self, sku, qty):
        if qty <= 0:
            raise ValueError("qty must be positive")
        self._stock[sku] = self._stock.get(sku, 0) + qty

    def remove(self, sku, qty):
        have = self._stock.get(sku, 0)
        if qty <= 0 or qty > have:
            raise ValueError(f"cannot remove {qty} of {sku} (have {have})")
        if have == qty:
            del self._stock[sku]
        else:
            self._stock[sku] = have - qty

    def __getitem__(self, sku):
        return self._stock.get(sku, 0)

    def __contains__(self, sku):
        return sku in self._stock

    def __len__(self):
        return len(self._stock)

    def __iter__(self):
        return iter(sorted(self._stock.items()))

    @contextmanager
    def batch(self):
        snapshot = dict(self._stock)
        try:
            yield self
        except BaseException:
            self._stock = snapshot
            raise

**Explanation**

Dunder methods make the object behave like a built-in container; a snapshot + `@contextmanager` gives rollback-on-error in a few lines.

**Complexity:** O(1) per op; O(n) snapshot.

**Common mistakes**

- Swallowing the exception in the context manager.